# AI PCB Inspector — full-board training (Colab only)

This notebook owns **all** development, training and export. Your PC/Pi only downloads a run bundle and runs `test_board.py` on a whole-board image.

What changed from the old pilot:
- The model accepts a **full board image**, finds joints itself, and reports boxes + labels. No caller-supplied crops.
- SolDef `no_good` (component misplacement) is mapped to `other`, **never** to `solder_bridge`. Unreviewed mappings cannot silently become ground truth.
- If there is no reviewed bridge source, the `solder_bridge` class is dropped from `labels.txt` so the model physically cannot call a bridge.
- Data source: Kaggle SolDef_AI only. No Roboflow workspace/project is used; an API key is ignored unless a project is configured in the config.
- The Kaggle archive ships labelme JSONs; `download` converts them once to `_annotations.coco.json` under `data_cache/soldef/`.

**Secrets to add in Colab Secrets (left sidebar 🔑), never in a cell or chat:** `KAGGLE_USERNAME`, `KAGGLE_KEY`, `GH_TOKEN` (PAT with write access). `ROBOFLOW_API_KEY` is not needed.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/AI-PCB-Inspector/repo')
assert REPO.is_dir(), 'Repo folder missing in Drive'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Repo:', REPO)
print('Colab-only workflow: no local sync step.')

In [ ]:
!pip -q install -U ultralytics kaggle tensorflow
import torch, tensorflow as tf, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('tensorflow', tf.__version__, 'ultralytics', ultralytics.__version__)

In [ ]:
# Pull secrets from Colab Secrets into the environment for the CLI.
# Nothing is printed except secret *names*. Do NOT paste keys into this notebook.
for name in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'ROBOFLOW_API_KEY', 'GH_TOKEN', 'GITHUB_TOKEN'):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f'{name}: set')
    else:
        print(f'{name}: missing (Kaggle + GH_TOKEN required; Roboflow unused)')

## 1. Download + visual label audit

Only the Kaggle SolDef source is configured; there is no Roboflow workspace/project, so the download step skips Roboflow even if an API key is present. The 428 per-image labelme JSONs are converted once into `data_cache/soldef/_annotations.coco.json`, pointing at the `Dataset/CS*/R*/V*/Setup*` copies so splits can group by board family.

Contact sheets appear under `runs/audit/<source>/<class>_contact.jpg`. Open them and decide which mappings are defensible. Edit `training/configs/fullboard_config.json`: set `"reviewed": true` only for mappings you can defend with an example image. Leave `solder_bridge` unmapped unless you see actual pad-to-pad bridging.

In [ ]:
import json, subprocess
from datetime import datetime, timezone
from pathlib import Path

CMD = 'python training/fullboard_colab.py'
CFG = 'training/configs/fullboard_config.json'
LOG_DIR = Path('/content/drive/MyDrive/AI-PCB-Inspector/runs/logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(*args):
    """Run a pipeline step, tee output to Drive, and show the real error on failure."""
    command = CMD.split() + [str(arg) for arg in args]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%SZ')
    log = LOG_DIR / f'{stamp}_{args[0]}.log'
    print('+', ' '.join(command), flush=True)
    print(f'Log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        handle.write('+ ' + ' '.join(command) + '\n')
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{args[0]} failed (exit {return_code}); full output in {log}')


run('download', '--config', CFG)
run('audit', '--config', CFG, '--previews', 8)

from IPython.display import display, Image as DisplayImage
audit_dir = Path('/content/drive/MyDrive/AI-PCB-Inspector/runs/audit')
summary = json.loads((audit_dir / 'audit_summary.json').read_text())
for source, entries in summary['sources'].items():
    print(f'{source}: {len(entries)} annotation file(s), '
          f'{sum(entry["images"] for entry in entries)} images')
sheets = sorted(audit_dir.rglob('*_contact.jpg'))
if not sheets:
    raise RuntimeError('No contact sheets generated; check the download/audit output above.')
for sheet in sheets:
    print(sheet.relative_to(audit_dir))
    display(DisplayImage(filename=str(sheet)))

## 2. Train + export + publish

`ALLOW_PROVISIONAL = True` lets the pipeline run before every mapping is reviewed. The run is still stamped with a `label_semantics` warning and the model card says which mappings are provisional. Set it to `False` once the audit is signed off.

**Step logs:** output from every step is now streamed to the cell and mirrored under `runs/logs/<UTC>_<step>.log` in Drive, so a failed export shows its real traceback and survives a session reset. If `export` fails but `train` succeeded, fix the cause and rerun only the `export` line below; the trained files in `WORK` are kept.


In [ ]:
WORK = '/content/pcb_fullboard_run'
ALLOW_PROVISIONAL = True
DEVICE = 'auto'
prepare_args = ['prepare', '--config', CFG, '--output', WORK]
if ALLOW_PROVISIONAL:
    prepare_args.append('--allow-provisional')
run(*prepare_args)
run('train', '--work', WORK, '--device', DEVICE)
run('export', '--work', WORK)

In [ ]:
# Publish the bundle to Drive handoff state and push the cleaned repo to GitHub.
import glob
run_dirs = sorted(glob.glob('/content/drive/MyDrive/AI-PCB-Inspector/runs/*/model_card.json'))
latest = max(run_dirs, key=os.path.getmtime)
latest_dir = str(Path(latest).parent)
run('publish', '--run-dir', latest_dir)
run('push', '--message', 'Colab: full-board inspector overhaul')

In [ ]:
# Verify the stable export bundle before downloading it locally.
import hashlib
current = Path('/content/drive/MyDrive/AI-PCB-Inspector/exports/current')
manifest = (current / 'manifest.sha256').read_text().splitlines()
for line in manifest:
    digest, name = line.split('  ', 1)
    actual = hashlib.sha256((current / name).read_bytes()).hexdigest()
    assert actual == digest, f'CHECKSUM MISMATCH: {name}'
print('Bundle OK:', current)
print('Files:', sorted(p.name for p in current.iterdir()))
print(json.dumps(json.loads((current / 'model_card.json').read_text())['classes'], indent=2))